# Train Test Split
Split the 25-feature labelled dataset into train and test sets.
Split is done at patient level to prevent data leakage.
All recordings from a patient, including gaussian augmented versions, go to the same split.
Stratified 80/20 split preserves the absent/present ratio in both sets.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT  = Path(r"D:\sop")
FEATURES_DIR  = PROJECT_ROOT / "data" / "features"
SPLITS_DIR    = PROJECT_ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE  = 42
TEST_SIZE     = 0.20

print("Ready.")

Ready.


In [2]:
# loading the selected features dataset

df = pd.read_csv(FEATURES_DIR / "features_selected.csv")

print(f"Loaded : {len(df)} recordings")
print(f"Classes: {df['class'].value_counts().to_dict()}")
print(f"Features: {len(df.columns) - 2}")

Loaded : 4782 recordings
Classes: {'absent': 2391, 'present': 2391}
Features: 25


In [3]:
# extracting patient ID from filename
# everything before the first underscore is the patient ID
# examples:
#   a100_AV_decomposed           ->  a100
#   p100_AV_gauss1_decomposed    ->  p100
#   p100_AV_gauss617_decomposed  ->  p100
# this groups all recordings and all gaussian augmented versions under one patient

df["patient_id"] = df["file"].str.split("_").str[0]

print(f"Unique patients: {df['patient_id'].nunique()}")
print(f"\nRecordings per patient (sample):")
print(df.groupby("patient_id").size().describe().to_string())

Unique patients: 874

Recordings per patient (sample):
count    874.000000
mean       5.471396
std        4.469034
min        1.000000
25%        3.000000
50%        4.000000
75%        4.000000
max       24.000000


In [4]:
# building patient level summary for the split
# each patient gets one row: their ID and their class label
# confirming no patient has conflicting labels across recordings

patient_df = df.groupby("patient_id")["class"].nunique().reset_index()
patient_df.columns = ["patient_id", "n_classes"]
conflicted = patient_df[patient_df["n_classes"] > 1]

if len(conflicted) > 0:
    print(f"ERROR: {len(conflicted)} patients have conflicting labels:")
    print(conflicted)
else:
    print("All patients have consistent labels across their recordings.")

# one label per patient
patient_labels = df.groupby("patient_id")["class"].first().reset_index()
patient_labels.columns = ["patient_id", "class"]

print(f"\nPatient level class distribution:")
print(patient_labels["class"].value_counts().to_string())
print(f"\nNote: imbalance at patient level is expected because gaussian")
print(f"augmentation was applied to present recordings to balance recording counts.")

All patients have consistent labels across their recordings.

Patient level class distribution:
class
absent     695
present    179

Note: imbalance at patient level is expected because gaussian
augmentation was applied to present recordings to balance recording counts.


In [5]:
# splitting patients 80/20
# stratified by class so both splits get proportional absent/present patients

train_patients, test_patients = train_test_split(
    patient_labels["patient_id"],
    test_size     = TEST_SIZE,
    stratify      = patient_labels["class"],
    random_state  = RANDOM_STATE
)

train_patients = set(train_patients)
test_patients  = set(test_patients)

print(f"Train patients: {len(train_patients)}")
print(f"Test patients : {len(test_patients)}")

Train patients: 699
Test patients : 175


In [6]:
# assigning all recordings to their patient split
# all recordings from a patient, including all gaussian versions, follow that patient

df_train = df[df["patient_id"].isin(train_patients)].copy().reset_index(drop=True)
df_test  = df[df["patient_id"].isin(test_patients)].copy().reset_index(drop=True)

# drop the patient_id column, no longer needed
df_train = df_train.drop(columns=["patient_id"])
df_test  = df_test.drop(columns=["patient_id"])

print(f"Train recordings: {len(df_train)}")
print(f"  absent : {len(df_train[df_train['class']=='absent'])}")
print(f"  present: {len(df_train[df_train['class']=='present'])}")
print(f"\nTest recordings : {len(df_test)}")
print(f"  absent : {len(df_test[df_test['class']=='absent'])}")
print(f"  present: {len(df_test[df_test['class']=='present'])}")
print(f"\nTotal check     : {len(df_train) + len(df_test)} (should be {len(df)})")

Train recordings: 3785
  absent : 1894
  present: 1891

Test recordings : 997
  absent : 497
  present: 500

Total check     : 4782 (should be 4782)


In [7]:
# verifying no patient appears in both train and test

train_pids = set(df_train["file"].str.split("_").str[0])
test_pids  = set(df_test["file"].str.split("_").str[0])
overlap    = train_pids.intersection(test_pids)

if len(overlap) == 0:
    print("No patient overlap between train and test. Split is clean.")
else:
    print(f"ERROR: {len(overlap)} patients found in both splits: {overlap}")

No patient overlap between train and test. Split is clean.


In [8]:
# saving train and test splits

train_out = SPLITS_DIR / "train.csv"
test_out  = SPLITS_DIR / "test.csv"

df_train.to_csv(train_out, index=False)
df_test.to_csv(test_out,  index=False)

print(f"Saved: {train_out}")
print(f"Saved: {test_out}")
print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape : {df_test.shape}")

Saved: D:\sop\data\splits\train.csv
Saved: D:\sop\data\splits\test.csv

Train shape: (3785, 27)
Test shape : (997, 27)
